## 🚚 Resource and Operational Optimization for Maintenance Logistics

This notebook demonstrates a **real-world resource and operational optimization** problem arising in reliability and maintenance.

### 🎯 Learning Objectives

- Formulate an optimization model
- Understand decision variables and constraints
- Solve using:
  - ✅ SciPy (`scipy.optimize.linprog`)
  - ✅ Gurobi
  - ✅ CasADi (NLP/Opti interface)

## 📖 Problem Statement

A company must assign maintenance work for **4 assets** over **4 days**.

Each asset requires exactly one maintenance task.

Each day has a limited maintenance capacity (crew-hours).

Scheduling an asset later increases the expected failure risk (predicted by a predictive maintenance model).

The goal is to **minimize total maintenance + expected failure costs**.

### 🧮 Mathematical Formulation

#### Sets

- $i\in I$: assets
- $t\in T$: days

#### Parameters

- $c_i^m$: maintenance cost
- $c_i^f$: failure cost
- $p_{it}$: probability of failure if maintained on day $t$
- $h_i$: crew-hours required
- $H_t$: available crew-hours on day $t$

#### Decision Variables

$
x_{it}=
\begin{cases}
1 & \text{asset }i\text{ maintained on day }t\\
0 & \text{otherwise}
\end{cases}
$

#### Objective

$
\min \sum_{i}\sum_{t}
\left(c_i^m+p_{it}c_i^f\right)x_{it}
$

#### Constraints

Each asset scheduled once

$
\sum_t x_{it}=1
$

Daily capacity

$
\sum_i h_i x_{it}\le H_t
$

Binary variables

$
x_{it}\in\{0,1\}
$

💡 In practice, $p_{it}$ comes from an ML/RUL prediction model, while optimization prescribes the best schedule.


In [9]:
import numpy as np

assets=['A1','A2','A3','A4']
days=[1,2,3,4]

maint_cost=np.array([200,250,180,300])
failure_cost=np.array([5000,4500,6000,5500])
hours=np.array([3,2,4,3])
capacity=np.array([5,5,5,5])

prob=np.array([
[0.03,0.05,0.09,0.15],
[0.02,0.04,0.08,0.12],
[0.04,0.06,0.10,0.18],
[0.03,0.05,0.07,0.14]
])

cost=maint_cost[:,None]+prob*failure_cost[:,None]
cost

array([[ 350.,  450.,  650.,  950.],
       [ 340.,  430.,  610.,  790.],
       [ 420.,  540.,  780., 1260.],
       [ 465.,  575.,  685., 1070.]])

### 1️⃣ Solution with SciPy (Linear Programming Relaxation)

In [2]:

# !pip install scipy

from scipy.optimize import linprog
import numpy as np

nA=len(assets)
nD=len(days)

cvec=cost.flatten()

Aeq=[]
beq=[]

# Each asset exactly once
for i in range(nA):
    row=np.zeros(nA*nD)
    for t in range(nD):
        row[i*nD+t]=1
    Aeq.append(row)
    beq.append(1)

Aub=[]
bub=[]

# Capacity constraints
for t in range(nD):
    row=np.zeros(nA*nD)
    for i in range(nA):
        row[i*nD+t]=hours[i]
    Aub.append(row)
    bub.append(capacity[t])

bounds=[(0,1)]*(nA*nD)

res=linprog(cvec,A_ub=Aub,b_ub=bub,A_eq=Aeq,b_eq=beq,bounds=bounds,method="highs")

print("Status:",res.message)
print("Optimal LP objective:",res.fun)
print("Decision matrix (fractional because LP relaxation):")
print(res.x.reshape((nA,nD)))


Status: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Optimal LP objective: 1875.0
Decision matrix (fractional because LP relaxation):
[[ 0.66666667  0.33333333  0.          0.        ]
 [ 1.          0.          0.          0.        ]
 [-0.          1.          0.          0.        ]
 [ 0.33333333  0.          0.66666667  0.        ]]


### 2️⃣ Solution with Gurobi (MILP)

In [5]:
# !pip install gurobipy
#
# Requires a valid Gurobi license.

try:
    import gurobipy as gp
    from gurobipy import GRB

    m=gp.Model()

    x=m.addVars(len(assets),len(days),vtype=GRB.BINARY,name="x")

    m.setObjective(
        gp.quicksum(cost[i,t]*x[i,t]
                    for i in range(len(assets))
                    for t in range(len(days))),
        GRB.MINIMIZE)

    for i in range(len(assets)):
        m.addConstr(gp.quicksum(x[i,t] for t in range(len(days)))==1)

    for t in range(len(days)):
        m.addConstr(
            gp.quicksum(hours[i]*x[i,t]
                        for i in range(len(assets)))
            <=capacity[t])

    m.optimize()

    if m.Status==GRB.OPTIMAL:
        print("\nOptimal schedule")
        for i,a in enumerate(assets):
            for t,d in enumerate(days):
                if x[i,t].X>0.5:
                    print(a,"-> Day",d)

        print("Objective =",m.ObjVal)

except Exception as e:
    print(e)

Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-1355U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 8 rows, 16 columns and 32 nonzeros (Min)


Model fingerprint: 0x0a159764
Model has 16 linear objective coefficients
Variable types: 0 continuous, 16 integer (16 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+02, 1e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Found heuristic solution: objective 2265.0000000
Presolve time: 0.00s
Presolved: 8 rows, 16 columns, 32 nonzeros
Variable types: 0 continuous, 16 integer (16 binary)

Root relaxation: objective 1.915000e+03, 6 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0    1915.0000000 1915.00000  0.00%     -    0s

Explored 1 nodes (6 simplex iterations) in 0.04 seconds (0.00 work units)
Thread count was 12 (of 12 available processors)

Solution count 2: 1915 2265 

Optimal solution found (tolerance 1.00e-04)
Best objective 1.915000000000e+03,

## 📚 Discussion

### 🔍 Comparison

| Solver | Model Type | Strengths | Limitations |
|---|---|---|---|
| SciPy | Linear Programming | Free, easy to learn | No binary variables in `linprog`; solves LP relaxation |
| Gurobi | MILP | State-of-the-art commercial solver, exact binary optimization | Requires a license |

### 🌍 Real-World Applications

- 🏭 Factory maintenance scheduling
- ⚡ Power grid maintenance planning
- 🚆 Railway inspection scheduling
- 🌬️ Wind farm technician allocation
- ✈️ Aircraft maintenance logistics

### 💡 Extensions

- Multiple crews
- Spare-parts inventory constraints
- Technician skills
- Time windows
- Multi-objective optimization (cost, availability, reliability)
- Robust and stochastic optimization
- Digital twin integration
